In [0]:
from pyspark.sql import functions as F

bronze_orders = spark.table("brazilian_ecommerce.prod.bronze_orders")

staging_orders = (
    bronze_orders
    .withColumn("order_status", F.lower(F.col("order_status")))

    # cast to proper datetime / date types
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_date("order_estimated_delivery_date"))

    # hours between each step (null-safe: if either side is null, result is null)
    .withColumn(
        "hours_purchase_to_approved",
        (F.col("order_approved_at").cast("long") - F.col("order_purchase_timestamp").cast("long")) / 3600
    )
    .withColumn(
        "hours_approved_to_carrier",
        (F.col("order_delivered_carrier_date").cast("long") - F.col("order_approved_at").cast("long")) / 3600
    )
    .withColumn(
        "hours_carrier_to_customer",
        (F.col("order_delivered_customer_date").cast("long") - F.col("order_delivered_carrier_date").cast("long")) / 3600
    )
    .withColumn(
        "hours_estimated_to_delivered",
        (F.col("order_delivered_customer_date").cast("long") - F.col("order_estimated_delivery_date").cast("timestamp").cast("long")) / 3600
    )

    # delivery delay flag: only meaningful when the order actually has a delivered date
    .withColumn(
        "delivery_status",
        F.when(F.col("order_delivered_customer_date").isNull(), F.lit(None))
         .when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date").cast("timestamp"), F.lit("delayed"))
         .otherwise(F.lit("on_time"))
    )

    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
)

full_table_name = "brazilian_ecommerce.prod.staging_orders"

(
    staging_orders.write
    .mode("overwrite")
    .option("comment", "orders staging table, with hours between each step and delivery status")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN order_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_staging_orders PRIMARY KEY (order_id)
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT fk_staging_orders_customer_id
    FOREIGN KEY (customer_id)
    REFERENCES brazilian_ecommerce.prod.staging_customers (customer_id)
""")

print(f"Loaded {staging_orders.count()} rows into {full_table_name}")